# Relative Value Difference Bias Check

This notebook will recover parameters from simulated data of three groups: large left preference, neutral perference, and large right preference. The underlying hypothesis is that parameter recovery bias (high noise, low drift) seems to be a result of a structural bias in the novel simulation methodology. Without quantifying this bias, which is believed to be conditional on relative value differences, the following work will only investigate the issues and determine if the bias is skewed towards large absolute value relative value differences if the bias is positionally dependent (left fixation is more likely).

## Stratifying Data

Define 
$$
\Delta_i = RDV_i = V_{L,i} - V_{R,i}.
$$

Then
$$
\mathcal{T}_R(\tau) = \{i:\Delta_i\le -\tau\},\\
\mathcal{T}_0(\epsilon) = \{i:|\Delta_i|\le\epsilon\},\\
\mathcal{T}_L(\tau) = \{i:\Delta_i\ge \tau\},
$$
and set $\tau = 3.5$ and $\epsilon = 0.5$.

In [ ]:
# If the notebook is moved into /exploratory_notebooks, then uncomment the code below

# import sys, os

# sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import pandas as pd
from ast import literal_eval

df_raw = pd.read_csv('1ms_trial_data.csv')
df_raw['RT'] = df_raw['RT']*1000 # adjustment for RT
df_raw['fixation'] = df_raw['fixation'].apply(literal_eval)

to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
]

,sub_id,trial,hidden,avgWTP_left,avgWTP_right,choice,RT,fixation
0,329,1,True,1.00,5.0,right,1723.0,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ..."
1,329,2,True,3.00,5.0,left,2573.0,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,329,3,True,4.25,1.0,left,1775.0,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ..."
3,329,4,True,3.50,1.0,left,1903.0,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ..."
4,329,5,True,1.50,1.0,left,1863.0,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ..."


In [ ]:
large_rvd_threshold = 3.5
neutral_rvd_threshold = 0.5

right_pref_df = df.loc[df["avgWTP_left"] - df["avgWTP_right"] <= -large_rvd_threshold]
neutral_pref_df = df.loc[abs(df["avgWTP_left"] - df["avgWTP_right"] <= 0.5)]
left_pref_df = df.loc[df["avgWTP_left"] - df["avgWTP_right"] >= large_rvd_threshold]

A quick reference to model-free analysis, the average response times of trials with large preferences to either stimuli should be shorter than trials with neutral perferences.

In [19]:
import numpy as np

print(f'Trials with large right preference: {right_pref_df.shape[0]} (average RT: {np.mean(right_pref_df['RT']):.2f} ms)')
print(f'Trials with neutral preference: {neutral_pref_df.shape[0]} (average RT: {np.mean(neutral_pref_df['RT']):.2f} ms)')
print(f'Trials with large left preference: {left_pref_df.shape[0]} (average RT: {np.mean(left_pref_df['RT']):.2f} ms)')

Trials with large right preference: 698 (average RT: 1211.14 ms)
Trials with neutral preference: 12419 (average RT: 1677.19 ms)
Trials with large left preference: 658 (average RT: 1229.28 ms)
